# v10: What shift do the public scores support? (posterior over k)

**Version:** v10  
**Kaggle submission:** ready to upload on 22 Nov: `v10_bayes`, `v10_k2` (and the earlier `v9_k4`) → `outputs/submission_v10_*.csv`

**Question:** after nine public scores, how large is the failure-rate shift between depots (the odds factor *k*), and which decision should we upload? Instead of reading one score at a time, the whole set of scores is used jointly: for every candidate *k* we simulate what **each uploaded file would have scored**, then ask which *k* makes the observed vector of scores most plausible.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
HIST = {a: [f"s{i:03d}" for i in range(a, a + 10)] for a in (23, 37, 54, 80, 129, 154)}   # six hidden 10-bin histograms
HCOLS = {col for g in HIST.values() for col in g}

def shape(df):
    """Histogram-shape features: per-bin shares, mean/std bin, entropy, low/high-3 tail shares, log usage."""
    out = {"logU": np.log1p(df[HIST[54][:9]].sum(axis=1, min_count=9))}
    for a, g in HIST.items():
        g = g[:9] if a == 54 else g                        # test s063 is NaN after realignment
        x = df[g].to_numpy(float); tot = x.sum(1, keepdims=True)
        sh = np.where(tot > 0, x / np.where(tot > 0, tot, 1), np.nan); k = np.arange(len(g)); mean = (sh * k).sum(1)
        out.update({f"h{a}_sh{i}": sh[:, i] for i in range(len(g))})
        out[f"h{a}_mean"] = np.where(tot[:, 0] > 0, mean, np.nan)
        out[f"h{a}_std"] = np.where(tot[:, 0] > 0, np.sqrt(np.maximum((sh * k ** 2).sum(1) - mean ** 2, 0)), np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"h{a}_ent"] = np.where(tot[:, 0] > 0, -np.nansum(np.where(sh > 0, sh * np.log(sh), 0), 1), np.nan)
        out[f"h{a}_lo3"], out[f"h{a}_hi3"] = sh[:, :3].sum(1), sh[:, -3:].sum(1)
    return pd.DataFrame(out, index=df.index)

def ratios(df, cols):
    U = df[HIST[54][:9]].sum(axis=1, min_count=9).replace(0, np.nan)
    return df[[col for col in cols if col not in HCOLS]].div(U, axis=0).add_suffix("_perU")

In [4]:
from scipy.stats import multivariate_normal
CM9 = {**CM, "n_estimators": 1000, "max_depth": 2}                      # v9 cost model
Pl = Platt().fit(fit_oof(dev[S]), y)
p_te_raw = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
q = Pl.predict(p_te_raw)                                                # calibrated test probabilities (old-depot scale)
f_all = tr[tr.failure == 1]; lc_all = np.log(f_all.repair_cost.to_numpy())
m_te, resid = cost_model(CM9, pd.concat([f_all[S], shape(f_all)], axis=1), lc_all, pd.concat([te[S], shape(te)], axis=1))
K = 99
M = np.exp(m_te[:, None] + np.quantile(resid, (np.arange(K) + .5) / K)[None, :]); mu = M.mean(1)   # (vehicle, cost draw)
nte = len(te)

### 1. The generative model of a public score
For a true shift $k$: vehicle $i$ fails with probability $\text{shift}(q_i,k)$ ($q_i$ = calibrated probability, $\text{shift}(p,k)=kp/(1-p+kp)$), its repair cost is drawn from the v9 per-vehicle cost distribution, and the public score is the total cost of a random 30% of the test vehicles. All uploaded files are scored on the **same** subset and the **same** failures, so their scores are strongly correlated; the simulation captures that. A multivariate normal is fitted to the simulated score vector for each $k$, a relative model-error term `eps` is added to its covariance (our probabilities and cost model are not exact), and the observed vector is evaluated.

In [5]:
OBS = {"v2": 74933.1668, "v3": 27760.1404, "v3b": 29529.7466, "v4_platt": 23875.6666, "v5_raw": 26060.5718,
       "v5_platt": 22968.5508, "v9_k1": 19600.792, "v7_k4": 19067.165, "v9_em": 18058.5558}
files = {k: pd.read_csv(f"../submissions/submission_{k}.csv") for k in OBS}
Fm = np.column_stack([files[k].flag.to_numpy() for k in OBS]); Bm = np.column_stack([files[k].budget.to_numpy() for k in OBS])
obs = np.array(list(OBS.values())); R = 1200; rng = np.random.default_rng(SEED)
GRID = [1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 10, 12]
mean, cov = {}, {}
for k in GRID:
    p = shift(q, k); T = np.zeros((R, len(OBS)))
    for r in range(R):
        fail = rng.random(nte) < p
        c_sim = np.where(fail, M[np.arange(nte), rng.integers(0, K, nte)], 0.0)
        pub = rng.random(nte) < 0.3; cc = c_sim[pub][:, None]; Fp, Bp = Fm[pub], Bm[pub]
        T[r] = np.where(Fp == 1, INSPECT + UNDER * np.maximum(cc - Bp, 0) + OVER * np.maximum(Bp - cc, 0), cc).sum(0)
    mean[k], cov[k] = T.mean(0), np.cov(T.T)
sim = pd.DataFrame({k: mean[k] for k in GRID}, index=list(OBS)).round(-2).astype(int); sim.insert(0, "observed", obs.round(0).astype(int))
print("simulated mean public score of each uploaded file under each true k:"); print(sim.to_string())

simulated mean public score of each uploaded file under each true k:
          observed    1.5    2.0    2.5    3.0    3.5    4.0    4.5    5.0    5.5     6.0     7.0     8.0    10.0    12.0
v2           74933  45600  54300  62900  69300  75600  81600  88000  92500  98400  102200  111400  118700  133900  147400
v3           27760  21000  24600  28600  31800  34800  37700  41400  43900  47200   49800   55100   59900   70000   79300
v3b          29530  22500  26600  31300  34800  38200  41500  45300  48400  51800   54800   60800   65800   76800   87000
v4_platt     23876  19300  22100  25200  27500  29900  32200  35000  37000  39600   41600   45700   49600   57600   65300
v5_raw       26061  17900  21000  24700  27400  30200  33000  36000  38400  41600   43800   48700   53100   62300   71200
v5_platt     22969  16000  18400  21200  23200  25200  27300  29700  31400  33500   35400   39000   42500   49600   56100
v9_k1        19601  14400  16500  18900  20700  22500  24300  26400  27900  2

### 2. Posterior over k (uniform prior on the grid), and how sensitive it is to the model-error term

In [6]:
def posterior(eps):
    L = np.array([multivariate_normal(mean=mean[k], cov=cov[k] + np.diag((eps * mean[k]) ** 2), allow_singular=True).logpdf(obs) for k in GRID])
    w = np.exp(L - L.max()); return w / w.sum(), L
post = {}
for eps in (0.03, 0.06, 0.10, 0.15):
    w, L = posterior(eps); post[eps] = w; cdf = np.cumsum(w)
    print(f"eps {eps:.2f}: posterior mean k {np.dot(w, GRID):.2f}   90% interval [{GRID[int(np.searchsorted(cdf, .05))]}, {GRID[min(int(np.searchsorted(cdf, .95)), len(GRID) - 1)]}]   mode {GRID[int(w.argmax())]}")
W = post[0.06]
print("\nposterior over k (eps 0.06):", {k: round(float(x), 3) for k, x in zip(GRID, W)})
kb = GRID[int(W.argmax())]
z = (obs - mean[kb]) / np.sqrt(np.diag(cov[kb]) + (0.06 * mean[kb]) ** 2)
print(f"standardised residuals at k={kb} (all within ~1 = the model explains the nine scores):", dict(zip(OBS, np.round(z, 2))))

eps 0.03: posterior mean k 3.24   90% interval [2, 4.5]   mode 3
eps 0.06: posterior mean k 3.01   90% interval [2, 4.5]   mode 2.5
eps 0.10: posterior mean k 2.73   90% interval [2, 4]   mode 2.5
eps 0.15: posterior mean k 2.49   90% interval [1.5, 4]   mode 2

posterior over k (eps 0.06): {1.5: 0.008, 2: 0.131, 2.5: 0.267, 3: 0.267, 3.5: 0.171, 4: 0.096, 4.5: 0.035, 5: 0.017, 5.5: 0.006, 6: 0.002, 7: 0.0, 8: 0.0, 10: 0.0, 12: 0.0}
standardised residuals at k=2.5 (all within ~1 = the model explains the nine scores): {'v2': np.float64(1.02), 'v3': np.float64(-0.14), 'v3b': np.float64(-0.25), 'v4_platt': np.float64(-0.25), 'v5_raw': np.float64(0.24), 'v5_platt': np.float64(0.4), 'v9_k1': np.float64(0.2), 'v7_k4': np.float64(-0.03), 'v9_em': np.float64(-0.32)}


### 3. Decisions: the posterior-averaged probability, and the cost of each candidate under the posterior
For a fixed decision the expected cost is linear in $p$, so the Bayes decision under uncertainty about $k$ is made with $\bar p=\sum_k w_k\,\text{shift}(q,k)$. Expected cost per vehicle on the **whole** test set is computed exactly (not by simulation) for each true $k$.

In [7]:
def ec(fl, b, p):
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return float(np.where(fl == 1, INSPECT + p * fail + (1 - p) * OVER * b, p * mu).mean())
pbar = sum(w * shift(q, k) for w, k in zip(W, GRID))
bay, k2 = decide_pv(pbar, m_te, resid), decide_pv(shift(q, 2), m_te, resid)
rows = {"v10_bayes": bay, "v10_k2": k2}
for nm in ("v9_k1", "v9_k4", "v9_em", "v5_platt"):
    d = pd.read_csv(f"../submissions/submission_{nm}.csv"); rows[nm] = (d.flag.to_numpy(), d.budget.to_numpy())
truth = (2, 2.5, 3, 3.5, 4, 5, 6.5)
tab = pd.DataFrame({f"true {k}": {nm: ec(*fb, shift(q, k)) for nm, fb in rows.items()} for k in truth})
tab["posterior avg"] = {nm: sum(w * ec(*fb, shift(q, k)) for w, k in zip(W, GRID)) for nm, fb in rows.items()}
tab["vs v9_em"] = tab["posterior avg"] / tab.loc["v9_em", "posterior avg"] - 1
print(f"posterior mean k {np.dot(W, GRID):.2f}; mean posterior p on test {pbar.mean():.4f}\nexpected cost / vehicle on the whole test set:")
print(tab.round(3).to_string()); print("test flag rates:", {nm: round(float(fb[0].mean()), 4) for nm, fb in rows.items()})

posterior mean k 3.01; mean posterior p on test 0.0328
expected cost / vehicle on the whole test set:
           true 2  true 2.5  true 3  true 3.5  true 4  true 5  true 6.5  posterior avg  vs v9_em
v10_bayes   2.699     2.910   3.115     3.316   3.513   3.898     4.455          3.115    -0.051
v10_k2      2.664     2.913   3.155     3.392   3.623   4.073     4.720          3.155    -0.039
v9_k1       2.775     3.116   3.445     3.764   4.073   4.671     5.520          3.442     0.049
v9_k4       2.765     2.952   3.135     3.315   3.491   3.837     4.340          3.136    -0.044
v9_em       2.984     3.133   3.280     3.425   3.568   3.848     4.258          3.281     0.000
v5_platt    3.101     3.496   3.875     4.239   4.592   5.269     6.226          3.869     0.179
test flag rates: {'v10_bayes': 0.0893, 'v10_k2': 0.0801, 'v9_k1': 0.064, 'v9_k4': 0.0956, 'v9_em': 0.1124, 'v5_platt': 0.0629}


### 4. Files

In [8]:
write_submission(te.id, bay[0], bay[1], "v10_bayes")
write_submission(te.id, k2[0], k2[1], "v10_k2")

wrote outputs/submission_v10_bayes.csv: flagged 8.94% of test vehicles
wrote outputs/submission_v10_k2.csv: flagged 8.02% of test vehicles


## Results and decisions

- **The nine scores support k ≈ 3** (posterior mean 3.0, 90% interval 2-4.5; it moves between 2.5 and 3.2 as the model-error term goes from 0.15 to 0.03). All standardised residuals are within about ±1, so the simulation explains the whole set of scores. **The EM estimate (6.5) has essentially zero posterior mass** (EM assumes only the failure rate changed; the extra covariate shift between depots biases it upward, as suspected in v7), and k = 4 sits at the edge (probability ~0.10).
- **Decisions.** The posterior-averaged decision (`v10_bayes`, behaves like k ≈ 3.0, flags 8.9% of test) has the lowest expected cost: **5.1% below `v9_em`** (the best upload so far), 4.4% for `v9_k4`, 3.9% for `v10_k2`. If the public set behaves like the model, that is roughly **17,100-17,300** public points against 18,059.
- **Over-correcting is cheaper than under-correcting, but not free:** `v9_em` (k=6.53) costs about 5% more than the Bayes build in expectation; `v9_k1` (no shift) costs 4.9% *more than `v9_em`*, and `v5_platt` (old cost model, no shift) 17.9% more.

**Caveats**
- All of this rests on our own model of the test depot: failures ~ Bernoulli(shift(q, k)) with **one constant k for all vehicles** and the v9 cost model. If the extra failures are concentrated in some vehicle types, or repair costs are higher at the new depot (which would act like a higher k), the inferred k is an *effective* value, not a measured failure rate.
- The public subset is ~6,000 vehicles with few failures. The model-error term `eps` is a crude stand-in for everything the simulation gets wrong; the posterior shifts a little with it (above) but the conclusion that k is well below 6.5 does not.
- **Public scores cannot separate the close candidates** (`v10_bayes`, `v9_k4`, `v10_k2` differ by ~1-2% in expected cost, i.e. ~150-350 points, below the public noise). Uploading them anyway gives the score-vs-k curve (k = 1, 2, 3, 4, 6.5 with the v9 cost model) for the report, but the final choice should rest on the posterior, not on which of them scores lowest on the public subset.
- Remaining improvement is more likely in the models (cost model, failure ranking) than in fine-tuning k.